In [9]:
import os
import pandas as pd
import numpy as np

In [10]:
# ----------------------------- Config -----------------------------
INPUT_CSV = "datasets/ultravideo_long/long.csv"

# Reference output format (columns: video,prompt)
REFERENCE_CSV = "/work/nlp/hzhao/datasets/e2e-ttt-video/ttt_imagine_10k.csv"

# Where to write the filtered datasets (one per duration group).
OUTPUT_DIR = "/home/hzhao/ttt-imagine-diffsynth/data_utils/datasets/ultravideo_long"

# Duration split threshold (seconds): rows are grouped into < THRESHOLD and >= THRESHOLD.
DURATION_THRESHOLD = 20.0

# Score columns to combine. Each is normalized to a per-group relative score,
# then averaged into a single ranking score.
SCORE_COLS = ["vtss_score", "motion_score", "video_clip_score"]

# Column holding the caption used for the output prompt.
CAPTION_COL = "Detailed Description"

# Column to copy as the output data path.
PATH_COL = "clip_id"

# After ranking by the average relative score, keep this top fraction of each
# group (1.0 = keep all, just ranked). Set KEEP_TOP_N instead for an absolute count.
KEEP_TOP_FRAC = 1.0
KEEP_TOP_N = None


In [11]:
# ----------------------------- Load -----------------------------
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} rows from {INPUT_CSV}")
print("Columns:", list(df.columns))

# Drop rows missing a caption or any score we rank on.
required = [CAPTION_COL, PATH_COL, "duration"] + SCORE_COLS
df = df.dropna(subset=required).reset_index(drop=True)
print(f"{len(df)} rows after dropping nulls in {required}")
df[["duration"] + SCORE_COLS].describe()

Loaded 16597 rows from datasets/ultravideo_long/long.csv
Columns: ['clip_id', 'url', 'frame_width', 'frame_height', 'fps', 'start_frame', 'end_frame', 'total_frames', 'start_time', 'end_time', 'duration', 'vtss_score', 'motion_score', 'video_clip_score', 'Brief Description', 'Detailed Description', 'Background', 'Theme Description', 'Style', 'Shot Type', 'Camera Movement', 'Lighting', 'Video Atmosphere', 'Summarized Description']
16597 rows after dropping nulls in ['Detailed Description', 'clip_id', 'duration', 'vtss_score', 'motion_score', 'video_clip_score']


,duration,vtss_score,motion_score,video_clip_score
count,16597.000000,16597.000000,16597.000000,16597.000000
mean,30.929477,0.054241,8.053262,0.252503
std,112.485060,0.015379,12.261329,0.013077
min,10.010000,0.010002,0.100001,0.200280
25%,12.480000,0.046532,1.449855,0.244701
50%,16.066667,0.059837,3.266389,0.253297
75%,23.560000,0.066156,8.833343,0.261081
max,4862.257400,0.072916,99.680525,0.302598


In [12]:
# --------------------- Step 1: split by duration ---------------------
# Two groups: short (< threshold) and long (>= threshold).
groups = {
    "short": df[df["duration"] < DURATION_THRESHOLD].copy(),
    "long": df[df["duration"] >= DURATION_THRESHOLD].copy(),
}
for name, g in groups.items():
    print(f"{name:>5}: {len(g)} rows (duration {'<' if name=='short' else '>='} {DURATION_THRESHOLD}s)")

short: 10962 rows (duration < 20.0s)
 long: 5635 rows (duration >= 20.0s)


In [13]:
# ----------- Step 2: relative score per type, then average -----------
# The raw scores live on very different scales (vtss ~0.01-0.07, motion ~0.1-99,
# video_clip ~0.2-0.3), so we cannot average them directly. Within each duration
# group we min-max normalize every score column to [0, 1] -- this is the
# "relative score" (where the row sits between the group's worst and best for that
# metric). The mean of the relative scores is the combined ranking score.

def add_relative_scores(g):
    g = g.copy()
    rel_cols = []
    for col in SCORE_COLS:
        lo, hi = g[col].min(), g[col].max()
        rel = f"rel_{col}"
        # If the column is constant within the group, relative score is neutral (0.5).
        g[rel] = 0.5 if hi == lo else (g[col] - lo) / (hi - lo)
        rel_cols.append(rel)
    g["avg_relative_score"] = g[rel_cols].mean(axis=1)
    return g.sort_values("avg_relative_score", ascending=False).reset_index(drop=True)

groups = {name: add_relative_scores(g) for name, g in groups.items() if len(g) > 0}

for name, g in groups.items():
    print(f"=== {name} (top 5 by avg_relative_score) ===")
    display(g[[PATH_COL, "duration"] + SCORE_COLS
             + [f"rel_{c}" for c in SCORE_COLS] + ["avg_relative_score"]].head())

=== short (top 5 by avg_relative_score) ===


,clip_id,duration,vtss_score,motion_score,video_clip_score,rel_vtss_score,rel_motion_score,rel_video_clip_score,avg_relative_score
0,5cceb41b-d64d-461f-b230-cff770c9dbf3.mp4,10.710700,0.069223,86.776469,0.284479,0.941293,0.870416,0.822913,0.878207
1,ab8d444c-feb3-4e40-8772-2f99b5134929.mp4,12.000000,0.069251,99.408026,0.261588,0.941738,0.997264,0.599194,0.846065
2,8bdee17e-365f-4a20-bd3d-6cb1e1b0d8b7.mp4,10.083333,0.067182,96.655220,0.260127,0.908854,0.969620,0.584909,0.821128
3,e9947da8-030d-4cba-8237-c36bc261a3bc.mp4,11.866667,0.065944,96.070567,0.261310,0.889179,0.963748,0.596471,0.816466
4,ca194373-5751-4bef-b80f-b2406a439877.mp4,19.283333,0.067684,98.138638,0.256219,0.916838,0.984516,0.546719,0.816024


=== long (top 5 by avg_relative_score) ===


,clip_id,duration,vtss_score,motion_score,video_clip_score,rel_vtss_score,rel_motion_score,rel_video_clip_score,avg_relative_score
0,3bfce05e-de60-4974-8b21-e92531d7a7f8.mp4,28.294933,0.068498,80.235608,0.286864,0.929838,0.825042,0.881138,0.878673
1,fd7b91b6-348d-4832-afe4-f14c066e92a7.mp4,27.240000,0.068415,81.865775,0.284262,0.928513,0.841827,0.854176,0.874839
2,9c68b3c4-0ad7-43a4-aae6-2819db38f0c5.mp4,20.600000,0.067908,69.908172,0.275754,0.920457,0.718706,0.765995,0.801720
3,74b66867-6a28-419b-ae2c-9db6fe49216e.mp4,46.616667,0.069266,79.815653,0.263546,0.942051,0.820718,0.639471,0.800747
4,e7cf423f-5e71-400a-a72d-25c1cc1f39e0.mp4,25.842483,0.067159,74.036361,0.270810,0.908546,0.761212,0.714754,0.794838


In [14]:
# --------------- Step 3: rank-filter each group ---------------
# Groups are already sorted by avg_relative_score (best first). Keep the top
# fraction (or top-N) of each group.
def keep_top(g):
    if KEEP_TOP_N is not None:
        return g.head(KEEP_TOP_N)
    n = int(round(len(g) * KEEP_TOP_FRAC))
    return g.head(n)

filtered = {name: keep_top(g) for name, g in groups.items()}
for name, g in filtered.items():
    print(f"{name}: kept {len(g)} / {len(groups[name])} rows")

short: kept 10962 / 10962 rows
long: kept 5635 / 5635 rows


In [15]:
# --------------- Step 4: write in ttt_imagine format ---------------
# Target format (see REFERENCE_CSV): columns are exactly `video,prompt`.
#   video  <- clip_id   (copied as-is, per request)
#   prompt <- CAPTION_COL
ref_cols = list(pd.read_csv(REFERENCE_CSV, nrows=0).columns)
print("Reference columns:", ref_cols)  # ['video', 'prompt']

os.makedirs(OUTPUT_DIR, exist_ok=True)

def to_output(g):
    return pd.DataFrame({
        "video": g[PATH_COL].values,
        "prompt": g[CAPTION_COL].values,
    })

outputs = {name: to_output(g) for name, g in filtered.items()}

for name, out in outputs.items():
    assert list(out.columns) == ref_cols, f"{list(out.columns)} != {ref_cols}"
    data_len = len(out) // 1000 + 1
    path = os.path.join(OUTPUT_DIR, f"ultravideo_long_filtered_{name}_{data_len}k.csv")
    out.to_csv(path, index=False)
    print(f"Wrote {len(out)} rows -> {path}")


Reference columns: ['video', 'prompt']
Wrote 10962 rows -> /home/hzhao/ttt-imagine-diffsynth/data_utils/datasets/ultravideo_long/ultravideo_long_filtered_short_11k.csv
Wrote 5635 rows -> /home/hzhao/ttt-imagine-diffsynth/data_utils/datasets/ultravideo_long/ultravideo_long_filtered_long_6k.csv


In [16]:
# Combined file: the raw input dataset directly, keeping only clip_id + caption,
# renamed to the reference column names. No filtering, original row order.
raw = pd.read_csv(INPUT_CSV)
combined = (
    raw[[PATH_COL, CAPTION_COL]]
    .rename(columns={PATH_COL: "video", CAPTION_COL: "prompt"})
    .reset_index(drop=True)
)
assert list(combined.columns) == ref_cols, f"{list(combined.columns)} != {ref_cols}"
data_len = len(combined) // 1000 + 1
combined_path = os.path.join(OUTPUT_DIR, f"ultravideo_long_filtered_all_{data_len}k.csv")
combined.to_csv(combined_path, index=False)
print(f"Wrote {len(combined)} rows -> {combined_path}")
combined.head()

Wrote 16597 rows -> /home/hzhao/ttt-imagine-diffsynth/data_utils/datasets/ultravideo_long/ultravideo_long_filtered_all_17k.csv


,video,prompt
0,510d0f27-2ae2-407c-882a-ef26cadb2a79.mp4,The video captures a lively scene inside a gym...
1,5a2c3df6-ebcc-4097-9afc-1039c06b5aca.mp4,The video captures a serene and joyous outdoor...
2,866073b2-9255-4d5e-8c48-19cc2dce9811.mp4,The video features a man seated in an art stud...
3,dc7fba51-dfff-4311-a1ba-641c44d5a768.mp4,The video captures a serene outdoor scene feat...
4,7b135930-540a-4774-b60b-bbc234e151e6.mp4,The video captures a detailed process of water...
